In [ ]:
pip install --upgrade openai

In [1]:
from openai import OpenAI
import pandas as pd
import json
from tqdm.auto import tqdm
import re
import ast
tqdm.pandas(desc="Processing DataFrame")

## 匯入資料

In [2]:
# ===處理過後的新聞資料===
news_df = pd.read_excel('爬蟲程式檔/整理過後的新聞資料/新聞資料0520.xlsx')
news_df.tail()

,news,section,time,title,source,keywords,content
2478,moneyDJ,總體經濟,2024/05/20 13:12,高階產品需求旺，中砂今年營運有望逐季增長,NaN,NaN,報導半導體材料暨再生晶圓廠中砂(1560)4月營收5.43億元，月減1.17%、年增6.63...
2479,moneyDJ,總體經濟,2024/05/20 12:58,大聯大：財務成本持穩；增加垂直領域合作,NaN,NaN,報導IC通路商大聯大(3702)今年各項業務皆在成長軌道上，同時機會將大於挑戰；公司也針對營...
2480,moneyDJ,總體經濟,2024/05/20 12:52,"日本黃金價格破紀錄、期貨價首度衝破""1.2萬""",NaN,NaN,報導因國際金價揚升、加上日圓貶值，也帶動日本以日圓計價的黃金價格創下歷史新高紀錄，零售價格連...
2481,moneyDJ,總體經濟,2024/05/20 12:38,揭產業3大方向 賴總統：台灣有能力成經濟日不落國,NaN,NaN,報導總統賴清德今(20)日宣誓就職，在發表就職演說中，針對產業發展，他表示，未來政府會跟產...
2482,moneyDJ,總體經濟,2024/05/20 12:17,迎消暑需求+拉貨延續，應廣Q2~Q3營運看成長,NaN,NaN,報導MCU廠應廣(6716)表示，雖然消費性市場仍未全面復甦，但客戶拉貨力道有延續趨勢，預期...


In [40]:
# ===處理過後的新聞資料===
news_df = pd.read_excel('爬蟲程式檔/整理過後的新聞資料/鉅亨網100.xlsx')
news_df.dropna(subset=['title', 'time', 'content'], inplace=True, how='any')
news_df.tail()

,news,time,title,summary,content,stock,etf,market
95,鉅亨網,2024-06-03 19:11:26,中信00948B 6/12掛牌 00896估配息0.45元年化配息率破10%,「主動 + 被動新秀主動出擊」的中國信託投資級債券傘型基金募集買氣熱絡，債券生力軍「中國信託...,「主動 + 被動新秀主動出擊」的中國信託投資級債券傘型基金募集買氣熱絡，債券生力軍「中國信託...,"['00948B', '00896']",['TWS:00896:STOCK'],"[{'code': '00896', 'name': '中信綠能及電動車', 'symbol..."
96,鉅亨網,2024-06-03 19:07:21,〈COMPUTEX〉基辛格：AI如同25年前網路崛起 台灣是科技業的中心,英特爾 (INTC-US) 今 (3) 晚在寒舍艾美宴請供應鏈，執行長基辛格受訪表示，台灣是...,英特爾 (INTC-US) 今 (3) 晚在寒舍艾美宴請供應鏈，執行長基辛格受訪表示，台灣是...,"['2317', '2382', '2324', '2356', '3231', '4938...",[],"[{'code': '2317', 'name': '鴻海', 'symbol': 'TWS..."
97,鉅亨網,2024-06-03 19:00:46,森崴旗下SFE進軍國際離岸風電市場 向日本風電案場招商,森崴能源 (6806-TW) 旗下子公司寶崴海事工程 (SFE) 今 (3) 日指出，將進軍...,森崴能源 (6806-TW) 旗下子公司寶崴海事工程 (SFE) 今 (3) 日指出，將進軍...,['6806'],[],"[{'code': '6806', 'name': '森崴能源', 'symbol': 'T..."
98,鉅亨網,2024-06-03 19:00:05,蔡明彰觀點：美中關税戰 貨櫃、營建、成熟製程漁翁得利,NaN,今天報告的主題是「美中關税戰 貨櫃、營建、成熟製程漁翁得利」。\n\n萬寶投顧蔡明彰強調，台...,"['2382', '6669', '2317', '3231', '2376', '3324...",[],"[{'code': '2303', 'name': '聯電', 'symbol': 'TWS..."
99,鉅亨網,2024-06-03 18:59:28,裕慶-KY 6/4起上市前競拍每股底價120元,裕慶 - KY(6957-TW) 規劃於 6 月下旬上市掛牌配，配合上市前辦理現金增資發行新...,商用展示架、智能電動床機構件廠商裕慶 - KY(6957-TW) 規劃於 6 月下旬上市掛牌...,['6957'],[],NaN


In [4]:
# Convert index to string and filter rows where index ends with '89'
# filtered_df = news_df2[news_df2.index.astype(str).str.endswith('8')].copy()
# filtered_df.reset_index(drop=True, inplace=True)
# # filtered_df = filtered_df.head(10)
# filtered_df

,news,time,title,summary,content,stock,etf,market
0,鉅亨網,2024-05-01 10:40:03,電子五哥4月股價走弱 僅「這家」上揚且受三大法人青睞,時序進到 5 月，回顧電子五哥 4 月股價表現普遍走弱，包括廣達、緯創、英業達 ，以及仁寶 ...,時序進到 5 月，回顧電子五哥 4 月股價表現普遍走弱，包括廣達 (2382-TW)、緯創 ...,"['2382', '3231', '2356', '2324', '4938']",[],"[{'code': '2382', 'name': '廣達', 'symbol': 'TWS..."
1,鉅亨網,2024-05-01 06:25:54,〈貴金屬盤後〉Fed會議在即 市場鎖住獲利 黃金下跌 但月線仍連3度上升,黃金價格周二 (30 日) 跌逾 1%，至一周低點，但強勁的避險需求和央行購買力道仍令金價連...,黃金價格周二 (30 日) 跌逾 1%，至一周低點，但強勁的避險需求和央行購買力道仍令金價連...,[],[],NaN
2,鉅亨網,2024-04-30 22:30:09,〈財報〉3M上季業績亮眼 計劃減少配息,3M 周二 (30 日) 美股盤前公布公布 2024 會計年度第一季財報，由於價格調漲和成本...,3M 周二 (30 日) 美股盤前公布公布 2024 會計年度第一季財報，由於價格調漲和成本...,['US-MMM'],[],"[{'code': 'MMM', 'name': '3M', 'symbol': 'USS:..."
3,鉅亨網,2024-05-12 09:10:04,白宮去中化再動作，拚寬頻基礎建設，台系網通股 中磊、啟碁、智邦、智易、正文 受惠,NaN,〈FCC 否決華為認證實驗室的延長許可申請〉/\n\n美中貿易戰以來，中國電信三雄遭美國列為...,"['5388', '6285', '2345', '3596', '4906']",[],"[{'code': '5388', 'name': '中磊', 'symbol': 'TWS..."
4,鉅亨網,2024-05-11 09:40:06,〈熱門股〉陽明、萬海財報好成績 周漲逾2成 雙創1年來新高價,〈熱門股〉陽明、萬海周漲逾 2 成 雙創 1 年來新高價,貨櫃海運運價有撐，使航商第一季財報陸續繳出好成績，激勵陽明 (2609-TW)、萬海 (26...,"['2609', '2615']",[],"[{'code': '2609', 'name': '陽明', 'symbol': 'TWS..."
5,鉅亨網,2024-05-11 06:30:45,路透：拜登政府下週對半導體、太陽能設備課新關稅,《路透》週五 (10 日) 引述三位知情人士報導，美國總統拜登將於下週二將宣布針對戰略行業對...,《路透》週五 (10 日) 引述三位知情人士報導，美國總統拜登將於下週二將宣布針對戰略行業對...,[],[],NaN
6,鉅亨網,2024-05-18 22:33:35,歐美海運航線半個月漲價50% 「一櫃難求」還會持續多久？,5 月本是國際海運市場傳統淡季，但今年情況有些反常，4 月底以來，歐洲、美洲航線運價漲幅普遍...,5 月本是國際海運市場傳統淡季，但今年情況有些反常，4 月底以來，歐洲、美洲航線運價漲幅普遍...,[],[],NaN
7,鉅亨網,2024-05-18 11:00:06,〈熱門股〉藍天周漲42% 公告4月自結EPS 0.21元,藍天 (2362-TW) 本周在首季獲利亮眼、今年筆電出貨、展望樂觀下，本周股價多日漲停，最...,藍天 (2362-TW) 本周在首季獲利亮眼、今年筆電出貨樂觀、AI PC 題材帶動下，本周...,['2362'],[],"[{'code': '2362', 'name': '藍天', 'symbol': 'TWS..."
8,鉅亨網,2024-05-18 07:11:01,〈貴金屬盤後〉黃金、銅價刷新歷史高點紀錄 白銀收登11年高點 但各家看法分歧,黃金和銅期貨周五 (17 日) 均上漲至歷史最高收盤價，而白銀上漲至 11 年高點，這些都有...,黃金和銅期貨周五 (17 日) 均上漲至歷史最高收盤價，而白銀上漲至 11 年高點，這些都有...,[],[],NaN
9,鉅亨網,2024-05-20 13:21:51,台灣首例！可成外資股東提案獲2家外資投票顧問機構支持,機殼廠可成 (2474-TW) 即將在 5 月 30 日召開股東會，台灣史上第一個外資股東提...,機殼廠可成 (2474-TW) 即將在 5 月 30 日召開股東會，台灣史上第一個外資股東提...,['2474'],[],"[{'code': '2474', 'name': '可成', 'symbol': 'TWS..."


In [35]:
company_code = pd.read_csv("公司代碼.csv")
company_code['公司代碼'] = company_code['有價證券代號及名稱\xa0'].apply(lambda x: x.split('　')[0])
company_code['公司代碼'] = company_code['公司代碼'].astype('int').astype('str')
company_code['公司名稱'] = company_code['有價證券代號及名稱\xa0'].apply(lambda x: x.split('　')[-1])
company_name_list = company_code['公司名稱'].value_counts().index
stock_num_list = company_code['公司代碼'].value_counts().index
company_name_dict = company_code.drop(columns=['有價證券代號及名稱\xa0', '國際證券辨識號碼(ISIN Code)', '上市日', '市場別', '產業別', 'CFICode',
       '備註',]).set_index('公司名稱')['公司代碼'].to_dict()
company_code_dict = company_code.drop(columns=['有價證券代號及名稱\xa0', '國際證券辨識號碼(ISIN Code)', '上市日', '市場別', '產業別', 'CFICode',
       '備註',]).set_index('公司代碼')['公司名稱'].to_dict()
company_code

,有價證券代號及名稱,國際證券辨識號碼(ISIN Code),上市日,市場別,產業別,CFICode,備註,公司代碼,公司名稱
0,1101 台泥,TW0001101004,1962/2/9,上市,水泥工業,ESVUFR,NaN,1101,台泥
1,1102 亞泥,TW0001102002,1962/6/8,上市,水泥工業,ESVUFR,NaN,1102,亞泥
2,1103 嘉泥,TW0001103000,1969/11/14,上市,水泥工業,ESVUFR,NaN,1103,嘉泥
3,1104 環泥,TW0001104008,1971/2/1,上市,水泥工業,ESVUFR,NaN,1104,環泥
4,1108 幸福,TW0001108009,1990/6/6,上市,水泥工業,ESVUFR,NaN,1108,幸福
...,...,...,...,...,...,...,...,...,...
1814,9949 琉園,TW0009949008,2003/11/21,上櫃,文化創意業,ESVUFR,NaN,9949,琉園
1815,9950 萬國通,TW0009950006,2004/2/17,上櫃,塑膠工業,ESVUFR,NaN,9950,萬國通
1816,9951 皇田,TW0009951004,2003/12/18,上櫃,電機機械,ESVUFR,NaN,9951,皇田
1817,9960 邁達康,TW0009960005,2004/12/6,上櫃,運動休閒,ESVUFR,NaN,9960,邁達康


In [73]:
# company_name_dict['戴爾']
company_code_dict['2454']

'聯發科'

In [36]:
# ===產業類別資歷料===
industry_sector_df = pd.read_csv('sector_data.csv')
industry_sector_df.dropna(inplace=True)
# industry_sector_df['公司代碼'] = industry_sector_df['公司代碼'].astype('int').astype('str')
indust_list = industry_sector_df['行業'].value_counts().index
# stock_num_list = industry_sector_df['公司代碼'].value_counts().index
# company_name_list = industry_sector_df['公司名稱'].value_counts().index
indust_list

Index(['半導體', '健康科技', '電子生產設備', '電子元件', '耐用消費品', '金融', '電子科技', '工業機械', '生產製造',
       '科技服務',
       ...
       '無線通訊', '煉油/銷售', '專業保險', '目錄/專業配銷', '電子/電器商店', '消費雜貨', '投資經理',
       '食物：肉/魚/乳製品', '主要電訊', '金融集團'],
      dtype='object', name='行業', length=120)

In [37]:
company_name_list

Index(['台泥', '泓格', '歐買尬', '元創精密', '榮昌', '家登', '德微', '康聯訊', '光耀', '安瑞-KY',
       ...
       '晶彩科', '嘉澤', '台勝科', '晶相光', '安馳', '柏騰', '華擎', '揚明光', '維熹', '有益'],
      dtype='object', name='公司名稱', length=1819)

## Prompt

In [ ]:
# ===API Key===
api_key = ""
client = OpenAI(api_key = api_key)

In [39]:
# ===Prompt file===
# 第一部分
with open('prompt_firsthalf.txt', 'r') as file:
    prompt1 = file.read()
## 產業資料
for i in range(len(indust_list)):
    if i == 0:
        prompt_indust_list = "'" + indust_list[i] + "', "
    elif i == len(indust_list) - 1:
        prompt_indust_list = prompt_indust_list + "'" + indust_list[i] + "'"
    else:
        prompt_indust_list = prompt_indust_list + "'" + indust_list[i] + "', "

# prompt1 = content.split("insert")[0] + prompt_indust_list + content.split("insert")[1] + content.split("insert")[2]
print(prompt1)

# 第二部分
with open('prompt_sechalf_m.txt', 'r') as file:
    prompt2_m = file.read()

# prompt2 = content.split("insert")[0]  + content.split("insert")[1] + content.split("insert")[2]
print(prompt2_m)

with open('prompt_sechalf_s.txt', 'r') as file:
    prompt2_s = file.read()

# prompt2 = content.split("insert")[0]  + content.split("insert")[1] + content.split("insert")[2]
print(prompt2_s)

你是一位專業的股市新聞分析師，請站在股票市場投資人的角度，根據以下步驟提供股票市場投資人會想要的資訊。我會給你新聞文章，你必須執行任務，且所有回應都為繁體中文，一共有六個步驟。

請特別注意，所有步驟皆不需說明理由，僅以json格式一次性回傳所有指定內容：

1. 步驟一：分辨文章內容是否會影響股市，對於投資者是有用的資訊。
若是這篇文章會影響到整體金融市場、產業與個股，請歸類為1；若沒有任何影響，請歸類為0。
 - 請以以下json格式回答，不需要給出任何理由:
{‘是否影響市場’ : 0/1}

2. 步驟二：分辨文章內容影響的範圍。
 - 任務說明：
請分辨這則新聞會影響的層面與範圍，若是這篇文章會影響到整體金融市場請歸類為’市場’(例如政府政策發表、原物料上漲等)；若是這篇文章會影響到某個股票請歸類為’股票’(新聞內容圍繞特定股票)。若沒有任何影響，則不需回傳。

 - 請以以下json格式回答，不需要給出任何理由:
{‘影響範圍’ : “市場”/”股票”}


3. 步驟三：分類屬於財金新聞或是時事新聞。
 - 任務說明：
請根據以下定義，判斷這篇文章是否是財金新聞或是這篇文章是否是時事新聞，若是財金新聞，請在’財金新聞’中回傳1，若是時事新聞，請在’時事新聞’回傳1。


定義：’’’
 - 財金新聞：
主要報導與金融市場、經濟動態、投資機會和企業財務狀況相關的資訊。通常包括市場分析、專家意見、財務報告和經濟數據解讀。常見的報導形式包括股票分析報告、經濟預測和投資建議。
 - 時事新聞：
涵蓋廣泛的社會、政治、文化、科技和環境等各方面的最新事件和話題。包括國內外政治動態、社會事件、科技進步、文化活動和環境問題。
‘’’
 - 請以以下json格式回答，不需要給出任何理由:
{‘財金新聞’: 0/1,
‘時事新聞’: 0/1}


4. 步驟四：如果新聞為財金新聞，請找出這則新聞涉及的股票市場。
 - 任務說明：
若新聞重點為財金相關，請從以下股票市場列表中找出新聞內容提及的所有股票市場，如果有多個以list呈現；如果新聞為時事新聞相關，則不需回傳。如果新聞內容未提及股票市場但新聞內容提及的公司主要是與某個股票市場相關，也請標注該股票市場，如果有台灣公司參與請優先標注台灣股票。

股票市場列表：’’’
[‘美國股票市場(美股)’, ‘日本股票市場(日股)’

In [56]:
def call_ChatGPT(prompt, client):
  completion = client.chat.completions.create(
    model="gpt-4",
    messages=[
      {"role": "assistant", "content": prompt}
    ]
  )
  return completion

def prompt_1st(prompt1, prompt_indust_list ,news, client):
    prompt = prompt1.split('insert')[0] + prompt_indust_list + prompt1.split('insert')[1] + news + prompt1.split('insert')[2]
    response = call_ChatGPT(prompt, client)
    return response

def prompt_sec(prompt2_m, prompt2_s, effect, effect_range, stock_market, trends, industry, company_list, stock_code, fund_list, etf_list, news, title ,client):
    try:
      if effect == None or pd.isna(effect):
          return None
      elif str(int(effect)) == '0':
        return None
      
      else:
        if effect_range == '市場':
          # print('市場', stock_market, trends, industry, company_list, stock_code, fund_list, etf_list)
          prompt = prompt2_m.split('insert')[0] + stock_market + prompt2_m.split('insert')[1] + trends + prompt2_m.split('insert')[2] + industry + prompt2_m.split('insert')[3] + company_list + prompt2_m.split('insert')[4] + stock_code + prompt2_m.split('insert')[5] + fund_list + prompt2_m.split('insert')[6] + etf_list + prompt2_m.split('insert')[7] + title+ prompt2_m.split('insert')[8] + news + prompt2_m.split('insert')[9]
          response = call_ChatGPT(prompt, client)
          return response
        
        elif effect_range == '股票':
          # print('股票', stock_market, trends, industry, company_list, stock_code, fund_list, etf_list)
          prompt = prompt2_s.split('insert')[0] + stock_market + prompt2_s.split('insert')[1] + trends + prompt2_s.split('insert')[2] + industry + prompt2_s.split('insert')[3] + company_list + prompt2_s.split('insert')[4] + stock_code + prompt2_s.split('insert')[5] + fund_list + prompt2_s.split('insert')[6] + etf_list + prompt2_s.split('insert')[7] + title+ prompt2_s.split('insert')[8] + news + prompt2_s.split('insert')[9]
          # print(prompt)
          response = call_ChatGPT(prompt, client)
          return response
        
        else:
          return None
    except:
       return None

def list_to_strings(lst):
  if isinstance(lst, list) == False:
     print(type(lst))
     return '無'
  else:
    if len(lst) == 0:
       return '無'
    else:
      try:
        if len(lst) == 1:
          prompt_list= "[" +  lst[0] + "]"
          return prompt_list
        
        for i in range(len(lst)):
          if i == 0:
            prompt_list = "[" +  lst[i]
          elif i+1 == len(lst):
            prompt_list = prompt_list  + ", " + lst[i] + "]"
          else:
            prompt_list = prompt_list  + ", " + lst[i]
          
        return prompt_list
      except:
        return '無'

In [42]:
# ===Iterate===
news_df['ChatGPT_response1'] = news_df.progress_apply(lambda x: prompt_1st(prompt1, prompt_indust_list, x['content'], client), axis=1)

Processing DataFrame:   0%|          | 0/100 [00:00<?, ?it/s]

In [53]:
news_df2.to_excel('鉅亨新聞100.xlsx')

In [51]:
# ===Interpret Response===

# Transfer to dict
def transfer_text_to_dict(text):
    try:
        text1 = json.loads(text.choices[0].message.json())['content'].replace("'", '"').replace('\n', '').replace("‘", '"').replace('’', '"').replace('“', '"').replace('‘’', '"')
        try:
            text2 = json.loads(text1)
            return text2
        except:
            return None
    except:
        return None

news_df['response1'] = news_df.apply(lambda x: transfer_text_to_dict(x['ChatGPT_response1']), axis=1)

# response to column
news_df['是否影響市場'] = news_df['response1'].apply(lambda x: int(x['是否影響市場']) if x else None)
news_df2 = news_df.dropna(subset='response1')
news_df2 = news_df2[news_df2['是否影響市場'] == 1]
news_df2.reset_index(drop =True, inplace=True)

news_df2['影響範圍'] = news_df2['response1'].apply(lambda x: x['影響範圍'] if x else None)
news_df2['財金新聞'] = news_df2['response1'].apply(lambda x: x['財金新聞'] if x  else None)
news_df2['時事新聞'] = news_df2['response1'].apply(lambda x: x['時事新聞'] if x else None)
news_df2['股票市場分類'] = news_df2['response1'].apply(lambda x: x['股票市場分類'] if x  else None)
news_df2['產業分類名稱'] = news_df2['response1'].apply(lambda x: x['產業分類名稱'] if x else None)
news_df2['公司名稱'] = news_df2['response1'].apply(lambda x: x['公司名稱'] if x else None)
news_df2['股票名稱'] = news_df2['response1'].apply(lambda x: x['股票名稱'] if x else None)
news_df2['基金名稱'] = news_df2['response1'].apply(lambda x: x['基金名稱'] if x  else None)
news_df2['ETF'] = news_df2['response1'].apply(lambda x: x['ETF'] if x or x['是否影響市場'] == 1 else None)
news_df2['時事新聞主題標籤'] = news_df2['response1'].apply(lambda x: x['時事新聞主題標籤'] if x else None)
# news_df2['市場重要性評分'] = news_df2['response1'].apply(lambda x: x['市場重要性評分'] if x else None)
# news_df2['股票重要性評分'] = news_df2['response1'].apply(lambda x: x['股票重要性評分'] if x else None)

# Check type of the response
news_df2['error_response'] = news_df2['response1'].apply(lambda x: False if x else True)

# 檢查是否是list
columns_to_check = ['股票市場分類', '產業分類名稱', '公司名稱', '股票名稱', '基金名稱', 'ETF', '時事新聞主題標籤']
def check_all_lists(row):
    return all(isinstance(row[col], list) for col in columns_to_check)
news_df2['isList'] = news_df2.apply(check_all_lists, axis=1)

# 檢查是否是dict
# columns_to_check = ['市場重要性評分', '股票重要性評分']
# def check_all_dicts(row):
#     return all(isinstance(row[col], dict) for col in columns_to_check)
# news_df2['isDict'] = news_df2.apply(check_all_dicts, axis=1)

# 產業標籤修改
news_df2['產業分類名稱_final'] = news_df2['產業分類名稱'].apply(lambda x: [element for element in x if element in indust_list] if type(x) == list else None)

# 股票代碼修改
news_df2['股票名稱2'] = news_df2['股票名稱'].apply(lambda x: [re.findall(r'(\d+)-TW', s) for s in x] if type(x) == list else None) ## 結尾有TW
news_df2['股票名稱2'] = news_df2['股票名稱2'].apply(lambda x: [num for sublist in x for num in sublist] if type(x) == list else None)
news_df2['股票名稱2'] = news_df2['股票名稱2'].apply(lambda x: [element for element in x if element in stock_num_list] if type(x) == list else None)
### 上櫃的似乎沒有

# 公司對照代碼
news_df2['公司名稱2'] = news_df2['公司名稱'].apply(lambda x: [re.sub(r'\(.*?\)', '', s).strip() for s in x] if type(x) == list else None)
news_df2['公司名稱2'] = news_df2['公司名稱2'].apply(lambda x: [element for element in x if element in company_name_list] if type(x) == list else None)
news_df2['公司代碼'] = news_df2['公司名稱2'].apply(lambda x: [company_name_dict[element] for element in x] if type(x) == list else None)

# 整合全部公司代碼
news_df2['公司代碼_final'] = news_df2.apply(lambda x: list(set(x['股票名稱2'] + x['公司代碼'])) if type(x['股票名稱2']) == list else None, axis = 1)

# 整合全部公司名稱
news_df2['公司_final'] = news_df2['公司代碼'].apply(lambda x: [company_code_dict[element] for element in x] if type(x) == list else None)
# 基金名稱
news_df2['基金名稱_final'] = news_df2['基金名稱'].apply(lambda x: [re.findall(r'(\d+)', s) for s in x] if type(x) == list else None)

# ETF
news_df2['ETF_final'] = news_df2['ETF'].apply(lambda x: [re.findall(r'(\d+)', s) for s in x] if type(x) == list else None)

# 評分

In [57]:
news_df2['ChatGPT_response2'] =news_df2.progress_apply(lambda x: prompt_sec(prompt2_m, prompt2_s, effect = x['是否影響市場'], effect_range= x['影響範圍'], stock_market=list_to_strings(x['股票市場分類']), trends=list_to_strings(x['時事新聞主題標籤']), industry=list_to_strings(x['產業分類名稱_final']), company_list=list_to_strings(x['公司_final']), stock_code=list_to_strings(x['公司代碼_final']), fund_list=list_to_strings(x['基金名稱_final']), etf_list=list_to_strings(x['ETF_final']), news=x['content'], title=x['title'], client=client) if x['response1'] else None, axis=1)

Processing DataFrame:   0%|          | 0/96 [00:00<?, ?it/s]

In [59]:
news_df2.to_excel('鉅亨新聞100-2.xlsx')

In [60]:
# Transfer to dict
news_df2['response2'] = news_df2.apply(lambda x: transfer_text_to_dict(x['ChatGPT_response2']), axis=1)

def dict_to_column(x, key):
    try:
        return x[key]
    except:
        return None

news_df2['股票市場重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, '股票市場重要性評分') if x else None) 
news_df2['時事新聞標籤重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, '時事新聞標籤重要性評分') if x else None) 
news_df2['產業類別重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, '產業類別重要性評分') if x else None) 
news_df2['公司重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, '公司重要性評分') if x else None) 
news_df2['股票重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, '股票重要性評分') if x else None) 
news_df2['基金重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, '基金重要性評分') if x else None) 
news_df2['ETF重要性評分'] = news_df2['response2'].apply(lambda x: dict_to_column(x, 'ETF重要性評分') if x else None)

news_df2['新聞標題'] = news_df2['response2'].apply(lambda x: x['新聞標題'] if x else None)

news_df2['重點一'] = news_df2['response2'].apply(lambda x: dict_to_column(x['重點摘要'], '重點一') if x else None)
news_df2['重點二'] = news_df2['response2'].apply(lambda x: dict_to_column(x['重點摘要'], '重點二') if x else None)
news_df2['重點三'] = news_df2['response2'].apply(lambda x: dict_to_column(x['重點摘要'], '重點三') if x else None)
news_df2['重點四'] = news_df2['response2'].apply(lambda x: dict_to_column(x['重點摘要'], '重點四') if x else None)
news_df2['重點五'] = news_df2['response2'].apply(lambda x: dict_to_column(x['重點摘要'], '重點五') if x else None)

In [74]:
news_df2['news_lenth'] = news_df2['content'].apply(lambda x: len(x))

In [75]:
news_df2

,news,time,title,summary,content,stock,etf,market,ChatGPT_response1,response1,...,股票重要性評分,基金重要性評分,ETF重要性評分,新聞標題,重點一,重點二,重點三,重點四,重點五,news_lenth
0,鉅亨網,2024-05-28 15:38:27,界霖今年營收估小增 銅價急漲將反映給客戶,導線架業者界霖 (5285-TW) 今 (28) 日召開法說會，協理暨發言人吳念誼表示，今年...,導線架業者界霖 (5285-TW) 今 (28) 日召開法說會，協理暨發言人吳念誼表示，今年...,['5285'],[],"[{'code': '5285', 'name': '界霖', 'symbol': 'TWS...",ChatCompletion(id='chatcmpl-9W3EZ9e4dRt4HA3eVB...,"{'是否影響市場': 1, '影響範圍': '股票', '財金新聞': 1, '時事新聞':...",...,{'5285': 4},None,None,隨著銅價飆漲界霖調整策略，今年營收有望小幅成長,界霖公司預計在今年下半年客戶需求回籠下，營收將逐步回升，尤其看好車用、工控業績且全年預估結果...,由於近期國際銅價急漲，界霖計劃將原料價格波動反映給客戶，同時也會密切關注實際需求並調整價格。,從產品的角度來看，界霖預計今年車用業績將年增5-10%，工控年增3-5%，而3C與消費性電子持平。,None,None,538
1,鉅亨網,2024-05-28 15:20:17,沃爾瑪四色ESL提前至Q2拉貨 元太、天鈺可望受惠,沃爾瑪 (Walmart) 原預計下半年開始大量導入四色電子貨架標籤 (ESL)，市場傳出，...,沃爾瑪 (Walmart)(WMT-US) 原預計下半年開始大量導入四色電子貨架標籤 (ES...,"['8069', '4961']",[],"[{'code': '4961', 'name': '天鈺', 'symbol': 'TWS...",ChatCompletion(id='chatcmpl-9W3EkuB4smd50OGxN3...,"{'是否影響市場': 1, '影響範圍': '股票', '財金新聞': 1, '時事新聞':...",...,"{'4961': 4, '8069': 4}",None,None,沃爾瑪四色ESL提前拉貨 加劇元太與天鈺之間的出貨競爭,沃爾瑪預計提前至今年第二季開始采購四色電子貨架標籤 (ESL)，元太(8069-TW) 與天...,元太已計劃停止生產三色ESL，並強力推銷四色ESL，並在價差明顯收斂的同時，開始大規模出貨。,天鈺作為元太的長期合作夥伴，與元太共同製造三色和四色ESL驅動IC。天鈺將從今年四色ESL銷...,天鈺預計今年第二季半導體成長最為明顯，其中四色ESL商品業績已超過七成。並且預計全年電子標籤...,None,433
2,鉅亨網,2024-05-28 15:20:04,高盛：全球石油需求將至少再成長十年才達峰值,高盛分析師表示，全球石油需求將至少再成長十年，他們預計電動車銷售動能放緩，將使石油產品需求保...,高盛分析師表示，全球石油需求將至少再成長十年，他們預計電動車銷售動能放緩，將使石油產品需求保...,[],[],NaN,ChatCompletion(id='chatcmpl-9W3EzrCjjXNZISBFhr...,"{'是否影響市場': 1, '影響範圍': '市場', '財金新聞': 1, '時事新聞':...",...,{'無': None},{'無': None},{'無': None},全球石油需求將在未來十年持續成長至峰值,根據高盛分析師預測，全球石油需求將至少保持在十年的成長期，預計到2034年達到峰值。,主要石油需求成長的國家將來自亞洲新興市場，特別是中國和印度，此預測與所有預測者的觀點相符。,電動車銷售的放緩是對石油需求成長的主要推手之一，因擔憂電動車的資本成本、政策不確定性及快速充...,None,None,672
3,鉅亨網,2024-05-28 15:18:36,金屬中心攜手尼得科 合作開發高效能馬達,金屬中心今 (28) 日宣布，與全球馬達龍頭日本尼得科 (Nidec)，共同簽署合作備忘錄 ...,金屬中心今 (28) 日宣布，與全球馬達龍頭日本尼得科 (Nidec)，共同簽署合作備忘錄 ...,[],[],NaN,ChatCompletion(id='chatcmpl-9W3F7080Ix5N7gClpE...,"{'是否影響市場': 1, '影響範圍': '市場', '財金新聞': 1, '時事新聞':...",...,{'無': 0},{'無': 0},{'無': 0},金屬中心聯手全球馬達龍頭尼得科開發高效能馬達 科技驅動工業變革,金屬中心與全球馬達龍頭日本尼得科簽署合作備忘錄，共同開發高效能馬達，初期鎖定工業節能馬達相關設備。,優勢互補，尼得科擅長高效率馬達設計，金屬中心擅長負載驗證及應用推廣。,此次合作將逐步引進尼得科自啟動磁阻馬達技術至台灣工業節能應用，並與台灣馬達產業協會合作，推廣...,None,None,621
4,鉅亨網,2024-05-28 15:15:36,〈台亞股東會〉李國光出任新董座 蔡育軒接任總座 集團亞家軍邁向IPO,台亞 (2340-TW) 今 (28) 日召開董事會，會中進行董事改選，會後董事會一致通過由...,台亞 (2340-TW) 今 (28) 日召開董事會，會中進行董事改選，會後董事會一致通過由...,"['2340', '7753']",[],"[{'code': '2340', 'name': '光磊', 'symbol': 'TWS...",ChatCompletion(id='chatcmpl-9W3FTT7aljKQnpIlDG...,"{'是否影響市場': 1, '影響範圍': '股票', '財金新聞': 1, '時事新聞':...",...,{'2340': 4},None,None,《新產品銷售與人事變動》台亞晉升新總座及董座，旗下冠亞新產品進入出貨階段,台亞公司進行董事改選，由李國光出任董事長，蔡育軒接任總經理。,台亞將積極與子公司合作，包括人才互通、資源共享、產能支援以及系統整合。,台亞表示即將開始出貨生產的 6 吋 GaN 功率產品及冠亞的 6 吋 SiC 功率產品，預計...,None,None,464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,鉅亨網,2024-06-03 19:11:26,中信00948B 6/12掛牌 00896估配息0.45元年化配息率破10%,「主動 + 被動新秀主動出擊」的中國信託投資級債券傘型基金募集買氣熱絡，債券生力軍「中國信託...,「主動 + 被動新秀主動出擊」的中國信託投資級債券傘型基金募集買氣熱絡，債券生力軍「中國信託...,"['00948B', '00896']",['TWS:00896:STOCK'],"[{'code': '00896', 'name': '中信綠能及電動車', 'symbol...",ChatCompletion(id='chatcmpl-9W3flDgJHX5Fnhvnfy...,"{'是否影響市場': 1, '影響範圍': '股票', '財金新聞': 1, '時事新聞':...",...,None,None,None,None,None,None,None,None,None,1212
92,鉅亨網,2024-06-03 19:07:21,〈COMPUTEX〉基辛格：AI如同25年前網路崛起 台灣是科技業的中心,英特爾 (INTC-US) 今 (3) 晚在寒舍艾美宴請供應鏈，執行長基辛格受訪表示，台灣是...,英特爾 (INTC-US) 今 (3) 晚在寒舍艾美宴請供應鏈，執行長基辛格受訪表示，台灣是...,"['2317', '2382', '2324', '2356', '3231', '4938...",[],"[{'code': '2317', 'name': '鴻海', 'symbol': 'TWS...",ChatCompletion(id='chatcmpl-9W3gEbazLu5gyaUgyW...,"{'是否影響市場': 1, '影響範圍': '股票', '財金新聞': 1, '時事新聞':...",...,None,None,None,None,None,None,None,None,None,500
93,鉅亨網,2024-06-03 19:00:46,森崴旗下SFE進軍國際離岸風電市場 向日本風電案場招商,森崴能源 (6806-TW) 旗下子公司寶崴海事工程 (SFE) 今 (3) 日指出，將進軍...,森崴能源 (6806-TW) 旗下子公司寶崴海事工程 (SFE) 今 (3) 日指出，將進軍...,['6806'],[],"[{'code': '6806', 'name': '森崴能源', 'symbol': 'T...",ChatCompletion(id='chatcmpl-9W3geVOFqqTg1ArxGT...,"{'是否影響市場': 1, '影響範圍': '股票', '財金新聞': 1, '時事新聞':...",...,Non

In [27]:
filtered_df.columns

Index(['news', 'time', 'title', 'summary', 'content', 'stock', 'etf', 'market',
       'ChatGPT_response1', 'response1', '是否影響市場', '影響範圍', '財金新聞', '時事新聞',
       '股票市場分類', '產業分類名稱', '公司名稱', '股票名稱', '基金名稱', 'ETF', '時事新聞主題標籤',
       'error_response', 'isList', '產業分類名稱_final', '股票名稱2', '公司名稱2', '公司代碼',
       '公司代碼_final', '公司_final', '基金名稱_final', 'ETF_final',
       'ChatGPT_response2', 'response2', '股票市場重要性評分', '時事新聞標籤重要性評分',
       '產業類別重要性評分', '公司重要性評分', '股票重要性評分', '基金重要性評分', 'ETF重要性評分', '新聞標題', '重點一',
       '重點二', '重點三', '重點四', '重點五'],
      dtype='object')

In [62]:
output_df = news_df2.drop(['ChatGPT_response1','response1','財金新聞', '時事新聞', '產業分類名稱', '公司名稱', '股票名稱', '基金名稱', 'ETF', 
       'error_response', 'isList',  'ChatGPT_response2', 'response2'], axis = 1)
output_df.to_excel('完成檔0603.xlsx', index = False)